# Section Classifier v0.1 — Local Training Notebook

Trains the resume section-splitter distillation model **entirely on local CPU**
(no Colab/GPU needed) — this only fits a lightweight classifier head on
*frozen* sentence embeddings, not a full transformer fine-tune, so it's a
minutes-scale job even on a laptop.

This notebook is a thin, interactive wrapper around the scripts already
built for this pipeline:
- `training/validate_section_dataset.py`
- `training/prepare_section_classifier_data.py`
- `training/fine_tune_section_classifier.py`
- `training/evaluate_section_classifier.py`

Run cells top to bottom. Re-run from step 2 onward any time the dataset
(`datasets/section_splitting/v0.1/labeled_lines.jsonl`) grows via
`scripts/generate_section_labeling_data.py --auto`.


## 0. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # in case the notebook is opened from notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

DATASET_VERSION = "v0.1"
LABELED_LINES_PATH = PROJECT_ROOT / "datasets" / "section_splitting" / DATASET_VERSION / "labeled_lines.jsonl"
CLASSIFIER_DATA_DIR = PROJECT_ROOT / "datasets" / "section_splitting" / DATASET_VERSION / "classifier"
MODEL_OUTPUT_DIR = PROJECT_ROOT / "models" / "section-classifier-v0.1"

print("Project root:", PROJECT_ROOT)
print("Labeled lines:", LABELED_LINES_PATH, "exists:", LABELED_LINES_PATH.exists())


## 1. Validate the dataset

Schema/label checks + label distribution (see `training/validate_section_dataset.py`).

In [ ]:
from training.validate_section_dataset import DatasetValidator, ALLOWED_LABELS

validator = DatasetValidator()
exit_code = validator.validate(LABELED_LINES_PATH)

if validator.warnings:
    print(f"{len(validator.warnings)} warning(s) (showing up to 10):")
    for w in validator.warnings[:10]:
        print(" -", w)

if validator.errors:
    print(f"
{len(validator.errors)} error(s):")
    for e in validator.errors[:20]:
        print(" -", e)
    raise SystemExit("Fix dataset errors before continuing.")

print(f"
Resumes: {validator.resume_count}   Labeled lines: {validator.line_count}")
for label in sorted(ALLOWED_LABELS, key=lambda l: -validator.label_counts[l]):
    count = validator.label_counts[label]
    pct = count / validator.line_count * 100 if validator.line_count else 0
    print(f"  {label:<16} {count:>6}  ({pct:.1f}%)")


### 1a. Label distribution chart

In [ ]:
import matplotlib.pyplot as plt

labels_sorted = sorted(ALLOWED_LABELS, key=lambda l: -validator.label_counts[l])
counts = [validator.label_counts[l] for l in labels_sorted]

plt.figure(figsize=(8, 4))
plt.bar(labels_sorted, counts)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Labeled lines")
plt.title(f"Section label distribution ({validator.resume_count} resumes, {validator.line_count} lines)")
plt.tight_layout()
plt.show()


## 2. Prepare train/validation/test splits

Deterministic 70/15/15 split by resume id (see `training/prepare_section_classifier_data.py`).

In [ ]:
from training.prepare_section_classifier_data import prepare

prepare(input_path=LABELED_LINES_PATH, output_dir=CLASSIFIER_DATA_DIR)


## 3. Train the classifier head

Embeds every line once (frozen `all-MiniLM-L6-v2`), fits a `LogisticRegression` head on embedding + structural features, and estimates the Viterbi label-transition matrix from train label sequences. Saves artifacts to `models/section-classifier-v0.1/`.

In [ ]:
from training.fine_tune_section_classifier import train as train_section_classifier

train_section_classifier(
    train_path=CLASSIFIER_DATA_DIR / "section_classifier_train.jsonl",
    validation_path=CLASSIFIER_DATA_DIR / "section_classifier_validation.jsonl",
    output_dir=MODEL_OUTPUT_DIR,
)


## 4. Evaluate on the held-out test split

Per-label precision/recall/F1 (with and without Viterbi smoothing) plus **boundary accuracy** — whether a section CHANGE is detected at the right line position, which matters more than raw per-line accuracy since labels are contiguous runs.

In [ ]:
from training.evaluate_section_classifier import evaluate

evaluate(
    test_path=CLASSIFIER_DATA_DIR / "section_classifier_test.jsonl",
    model_dir=MODEL_OUTPUT_DIR,
)


## 5. Qualitative check — inspect a few test resumes

Run the trained model on a handful of held-out resumes and compare its Viterbi-smoothed predictions to the LLM-teacher's true labels side by side.

In [ ]:
import json
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer

from app.ml.section_classifier_features import LABELS, align_proba_to_labels, compute_features, viterbi_decode

config = json.loads((MODEL_OUTPUT_DIR / "config.json").read_text(encoding="utf-8"))
embedder = SentenceTransformer(config["base_model"])
classifier = joblib.load(MODEL_OUTPUT_DIR / "classifier.joblib")
transition_log_probs = np.load(MODEL_OUTPUT_DIR / "transition_matrix.npy")

test_rows = [json.loads(l) for l in (CLASSIFIER_DATA_DIR / "section_classifier_test.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]

by_resume = {}
for r in test_rows:
    by_resume.setdefault(r["resume_id"], []).append(r)
for rows in by_resume.values():
    rows.sort(key=lambda r: r["line_index"])

N_SAMPLES = 3
for resume_id in list(by_resume)[:N_SAMPLES]:
    rows = by_resume[resume_id]
    texts = [r["text"] for r in rows]
    true_labels = [r["label"] for r in rows]

    X, _ = compute_features(rows, embedder)
    proba = align_proba_to_labels(classifier.predict_proba(X), classifier.classes_, LABELS)
    pred_indices = viterbi_decode(np.log(np.clip(proba, 1e-12, None)), transition_log_probs)
    pred_labels = [LABELS[i] for i in pred_indices]

    print(f"=== {resume_id} ===")
    for text, true, pred in zip(texts, true_labels, pred_labels):
        marker = "  " if true == pred else "!!"
        print(f"{marker} true={true:<15} pred={pred:<15} | {text[:70]}")
    print()


## 6. Next steps

- If the boundary accuracy / per-label F1 above look reasonable, enable the
  model in `.env`:
  ```
  SECTION_CLASSIFIER_MODEL_PATH=models/section-classifier-v0.1
  SECTION_CLASSIFIER_FALLBACK_MODE=model_with_regex_fallback
  ```
- Run `python scripts/compare_splitters.py --input datasets/section_splitting/v0.1/labeled_lines.jsonl`
  to see a line-level diff of how many lines move out of the regex splitter's
  `"other"` bucket once the model is enabled.
- Re-run this notebook from step 2 onward any time the dataset grows.
